# Lending Club Interactive Loan Default Predictor

This notebook loads trained artifacts and provides an interactive dashboard where a user can enter applicant/loan information and get:
 
1. Predicted probability of default
2. Threshold-based default flag
3. Visual risk gauge# Lending Club Interactive Loan Default Predictor

This notebook loads trained artifacts and lets users input applicant/loan details to predict default risk.

Outputs:
1. Predicted default probability
2. Threshold-based prediction (likely default vs non-default)
3. Risk band + gauge chart

Required:
- `artifacts/models/trained_models_bundle.joblib`
 
Required files:
- `artifacts/cleaning_only/cleaning_pipelines.joblib`
- `artifacts/models/trained_models_bundle.joblib`

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

display(HTML("""
<style>
.container { width: 98% !important; }
.jp-Notebook { max-width: 100% !important; }
</style>
"""))

In [2]:
# Load artifacts + model registry
MODEL_DIR = Path("artifacts/models")
TRAINED_BUNDLE_PATH = MODEL_DIR / "trained_models_bundle.joblib"

if not TRAINED_BUNDLE_PATH.exists():
    raise FileNotFoundError(f"Missing {TRAINED_BUNDLE_PATH}. Run updated data_modeling.ipynb first.")

trained_bundle = joblib.load(TRAINED_BUNDLE_PATH)

MODEL_SPECS = {
    "Logistic (Fundamental)": {
        "model_key": "log_fund",
        "encoder_key": "enc_fund_log",
        "threshold_key": "log_fund_val_best_f1_thr",
        "uses_full_fields": False,
    },
    "XGB (Fundamental)": {
        "model_key": "xgb_fund",
        "encoder_key": "enc_fund_xgb",
        "threshold_key": "xgb_fund_val_best_f1_thr",
        "uses_full_fields": False,
    },
    "XGB (Full)": {
        "model_key": "xgb_full",
        "encoder_key": "enc_full_xgb",
        "threshold_key": "xgb_full_val_best_f1_thr",
        "uses_full_fields": True,
    },
}

for name, spec in MODEL_SPECS.items():
    if spec["model_key"] not in trained_bundle:
        raise KeyError(f"{name}: missing model key '{spec['model_key']}'")
    if spec["encoder_key"] not in trained_bundle:
        raise KeyError(f"{name}: missing encoder key '{spec['encoder_key']}'")

print("Loaded trained bundle.")
print("Models:", list(MODEL_SPECS.keys()))

Loaded trained bundle.
Models: ['Logistic (Fundamental)', 'XGB (Fundamental)', 'XGB (Full)']


In [3]:
# Helpers: validation, feature build, prediction, metrics
FIXED_ISSUE_DATE = "Jan-2016"
CLAMP_OUT_OF_RANGE = False  # True => auto-clamp out-of-range values

def _safe_float(x):
    try:
        if x is None or (isinstance(x, str) and x.strip() == ""):
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def _norm_cat(x):
    if x is None:
        return np.nan
    s = str(x).strip()
    return np.nan if s == "" else s.lower()

def _term_to_num(term):
    if term is None:
        return np.nan
    m = re.search(r"(\d+)", str(term))
    return float(m.group(1)) if m else np.nan

def _emp_to_years(emp):
    if emp is None:
        return np.nan
    s = str(emp).strip().lower()
    if s == "":
        return np.nan
    s = re.sub(r"10\+", "10", s)
    s = re.sub(r"<\s*1", "0", s)
    m = re.search(r"(\d+)", s)
    return float(m.group(1)) if m else np.nan

def _parse_mon_yyyy(s, field_name):
    if s is None or str(s).strip() == "":
        return pd.NaT
    try:
        return pd.to_datetime(str(s).strip(), format="%b-%Y")
    except Exception:
        raise ValueError(f"{field_name} must be Mon-YYYY (example: Jan-2010).")

def _risk_band(p):
    if p < 0.05:
        return "Low"
    elif p < 0.15:
        return "Moderate"
    elif p < 0.30:
        return "Elevated"
    return "High"

def _fmt_metric(x):
    return f"{float(x):.3f}"

def _compute_derived_financials(monthly_debt, gross_monthly_income, revol_bal, revolving_credit_limit):
    md = _safe_float(monthly_debt)
    gi = _safe_float(gross_monthly_income)
    rb = _safe_float(revol_bal)
    rcl = _safe_float(revolving_credit_limit)

    if pd.isna(md) or pd.isna(gi) or pd.isna(rb) or pd.isna(rcl):
        raise ValueError("Monthly debt, monthly income, revolving balance, and revolving credit limit are required.")
    if gi <= 0:
        raise ValueError("Gross monthly income must be > 0.")
    if rcl <= 0:
        raise ValueError("Revolving credit limit must be > 0.")
    if md < 0 or rb < 0:
        raise ValueError("Monthly debt and revolving balance cannot be negative.")

    dti = (md / gi) * 100.0
    revol_util = (rb / rcl) * 100.0
    return float(dti), float(revol_util)

def _get_human_loop_metrics(model_name):
    summary = trained_bundle.get("summary", None)
    if summary is None or len(summary) == 0:
        raise ValueError("Missing summary in trained bundle. Re-run updated data_modeling.ipynb.")

    required_cols = [
        "model",
        "val_best_f1",
        "val_accuracy_at_best_f1_thr",
        "val_recall_at_best_f1_thr",
        "val_precision_at_best_f1_thr",
        "val_best_thr_f1",
    ]
    missing = [c for c in required_cols if c not in summary.columns]
    if missing:
        raise ValueError(f"Summary missing required columns: {missing}. Re-run updated data_modeling.ipynb.")

    row = summary.loc[summary["model"] == model_name]
    if row.empty:
        raise ValueError(f"No summary row for model: {model_name}")
    row = row.iloc[0]
    return {
        "f1": float(row["val_best_f1"]),
        "accuracy": float(row["val_accuracy_at_best_f1_thr"]),
        "recall": float(row["val_recall_at_best_f1_thr"]),
        "precision": float(row["val_precision_at_best_f1_thr"]),
        "threshold": float(row["val_best_thr_f1"]),
    }

BASE_RANGES = {
    "fico_range_low": (300, 900),
    "fico_range_high": (300, 900),
    "loan_amnt": (500, 40000),
    "annual_inc": (1000, 5_000_000),
    "open_acc": (0, 100),
    "total_acc": (0, 250),
    "delinq_2yrs": (0, 50),
    "pub_rec": (0, 50),
    "dti": (0, 100),
    "revol_bal": (0, 2_000_000),
    "revol_util": (0, 200),
}
FULL_MODEL_RANGES = {
    "int_rate": (0, 40),
    "installment": (1, 5000),
}

def _validate_range(field_name, value, lo, hi, clamp=False, warnings=None):
    if warnings is None:
        warnings = []
    v = _safe_float(value)
    if pd.isna(v):
        raise ValueError(f"{field_name} is required.")
    if v < lo or v > hi:
        if clamp:
            old_v = v
            v = float(np.clip(v, lo, hi))
            warnings.append(f"{field_name}={old_v} was clamped to {v}.")
        else:
            raise ValueError(f"{field_name} must be between {lo} and {hi}.")
    return v

def validate_form_by_model(model_name, form_data, clamp=False):
    f = dict(form_data)
    warnings = []

    for field, (lo, hi) in BASE_RANGES.items():
        f[field] = _validate_range(field, f.get(field), lo, hi, clamp=clamp, warnings=warnings)

    if model_name == "Logistic (Fundamental)":
        if not (300 <= f["fico_range_low"] <= 900):
            raise ValueError("FICO low must be between 300 and 900")
        if not (300 <= f["fico_range_high"] <= 900):
            raise ValueError("FICO high must be between 300 and 900")

    if f["fico_range_high"] < f["fico_range_low"]:
        if clamp:
            a, b = f["fico_range_low"], f["fico_range_high"]
            f["fico_range_low"], f["fico_range_high"] = b, a
            warnings.append("fico_range_high < fico_range_low; values swapped.")
        else:
            raise ValueError("fico_range_high must be >= fico_range_low.")

    if f["open_acc"] > f["total_acc"]:
        if clamp:
            f["total_acc"] = f["open_acc"]
            warnings.append("open_acc > total_acc; total_acc set to open_acc.")
        else:
            raise ValueError("open_acc cannot exceed total_acc.")

    issue_dt = _parse_mon_yyyy(f.get("issue_d"), "Issue date")
    earliest_dt = _parse_mon_yyyy(f.get("earliest_cr_line"), "Earliest credit line")
    if pd.notna(issue_dt) and pd.notna(earliest_dt) and earliest_dt > issue_dt:
        raise ValueError("earliest_cr_line must be <= issue_d.")

    if model_name == "XGB (Full)":
        for field, (lo, hi) in FULL_MODEL_RANGES.items():
            f[field] = _validate_range(field, f.get(field), lo, hi, clamp=clamp, warnings=warnings)

        grade = str(f.get("grade", "")).upper().strip()
        sub_grade = str(f.get("sub_grade", "")).upper().strip()
        if grade not in list("ABCDEFG"):
            raise ValueError("grade must be one of A-G.")
        if not re.fullmatch(r"[A-G][1-5]", sub_grade):
            raise ValueError("sub_grade must be in format A1..G5.")
        if sub_grade[0] != grade:
            raise ValueError("sub_grade must match grade initial.")

    return f, warnings

def _get_required_cols_from_encoder(encoder):
    if hasattr(encoder, "feature_names_in_"):
        return list(encoder.feature_names_in_)
    cols = []
    for name, trans, c in encoder.transformers_:
        if name == "remainder":
            continue
        if isinstance(c, (list, tuple, np.ndarray, pd.Index)):
            cols.extend(list(c))
    out, seen = [], set()
    for c in cols:
        if c not in seen:
            out.append(c)
            seen.add(c)
    if not out:
        raise ValueError("Could not infer expected input columns from encoder.")
    return out

def _get_num_cols_from_encoder(encoder):
    for name, trans, cols in encoder.transformers_:
        if name == "num":
            return list(cols)
    return []

def _build_input_row_for_encoder(encoder, form_data):
    req_cols = _get_required_cols_from_encoder(encoder)
    num_cols = _get_num_cols_from_encoder(encoder)

    row = {c: np.nan for c in req_cols}
    def put(k, v):
        if k in row:
            row[k] = v

    annual_inc = _safe_float(form_data.get("annual_inc"))
    open_acc = _safe_float(form_data.get("open_acc"))
    total_acc = _safe_float(form_data.get("total_acc"))
    delinq_2yrs = _safe_float(form_data.get("delinq_2yrs"))
    pub_rec = _safe_float(form_data.get("pub_rec"))
    dti = _safe_float(form_data.get("dti"))
    revol_bal = _safe_float(form_data.get("revol_bal"))
    revol_util = _safe_float(form_data.get("revol_util"))
    loan_amnt = _safe_float(form_data.get("loan_amnt"))
    int_rate = _safe_float(form_data.get("int_rate"))
    installment = _safe_float(form_data.get("installment"))
    term = _term_to_num(form_data.get("term"))
    emp_length_years = _emp_to_years(form_data.get("emp_length"))

    fico_low = _safe_float(form_data.get("fico_range_low"))
    fico_high = _safe_float(form_data.get("fico_range_high"))
    fico = np.nan if np.isnan(fico_low) or np.isnan(fico_high) else (fico_low + fico_high) / 2.0

    issue_dt = _parse_mon_yyyy(form_data.get("issue_d"), "Issue date")
    earliest_dt = _parse_mon_yyyy(form_data.get("earliest_cr_line"), "Earliest credit line")

    if pd.notna(issue_dt):
        issue_month = float(issue_dt.month)
        issue_weekday = float(issue_dt.weekday())
        issue_quarter = float(issue_dt.quarter)
        issue_weekofyear = float(issue_dt.isocalendar().week)
        issue_year = float(issue_dt.year)
    else:
        issue_month = issue_weekday = issue_quarter = issue_weekofyear = issue_year = np.nan

    if pd.notna(issue_dt) and pd.notna(earliest_dt):
        credit_age_years = (issue_dt - earliest_dt).days / 365.25
        if credit_age_years < 0 or credit_age_years > 100:
            credit_age_years = np.nan
    else:
        credit_age_years = np.nan

    credit_age_bucket = np.nan
    if pd.notna(credit_age_years):
        credit_age_bucket = pd.cut(
            pd.Series([credit_age_years]),
            bins=[-1, 1, 3, 5, 10, 20, 100],
            labels=False
        ).iloc[0]
        credit_age_bucket = float(credit_age_bucket) if pd.notna(credit_age_bucket) else np.nan

    revol_to_loan = np.nan
    if pd.notna(revol_bal) and pd.notna(loan_amnt) and loan_amnt != 0:
        revol_to_loan = revol_bal / loan_amnt

    installment_to_monthly_income = np.nan
    if pd.notna(installment) and pd.notna(annual_inc) and annual_inc != 0:
        installment_to_monthly_income = installment / (annual_inc / 12.0)

    open_to_total_ratio = np.nan
    if pd.notna(open_acc) and pd.notna(total_acc) and total_acc != 0:
        open_to_total_ratio = open_acc / total_acc

    fico_times_revolutil = np.nan
    if pd.notna(fico) and pd.notna(revol_util):
        fico_times_revolutil = fico * (revol_util / 100.0)

    revol_util_gt_100 = np.nan
    if pd.notna(revol_util):
        revol_util_gt_100 = float(revol_util > 100)

    put("annual_inc", annual_inc)
    put("open_acc", open_acc)
    put("total_acc", total_acc)
    put("delinq_2yrs", delinq_2yrs)
    put("pub_rec", pub_rec)
    put("dti", dti)
    put("revol_bal", revol_bal)
    put("revol_util", revol_util)
    put("loan_amnt", loan_amnt)
    put("term", term)
    put("int_rate", int_rate)
    put("installment", installment)
    put("fico", fico)
    put("credit_age_years", credit_age_years)
    put("emp_length_years", emp_length_years)

    put("issue_d_month", issue_month)
    put("issue_d_weekday", issue_weekday)
    put("issue_d_quarter", issue_quarter)
    put("issue_d_weekofyear", issue_weekofyear)
    put("issue_d_year", issue_year)

    put("revol_to_loan", revol_to_loan)
    put("installment_to_monthly_income", installment_to_monthly_income)
    put("open_to_total_ratio", open_to_total_ratio)
    put("fico_times_revolutil", fico_times_revolutil)
    put("revol_util_gt_100", revol_util_gt_100)
    put("credit_age_bucket", credit_age_bucket)

    put("home_ownership", _norm_cat(form_data.get("home_ownership")))
    put("verification_status", _norm_cat(form_data.get("verification_status")))
    put("purpose", _norm_cat(form_data.get("purpose")))
    put("addr_state", _norm_cat(form_data.get("addr_state")))
    put("grade", _norm_cat(form_data.get("grade")))
    put("sub_grade", _norm_cat(form_data.get("sub_grade")))

    for c in req_cols:
        if c.endswith("_missing"):
            base = c[:-8]
            row[c] = 1.0 if pd.isna(row.get(base, np.nan)) else 0.0

    X = pd.DataFrame([row], columns=req_cols)
    if num_cols:
        X[num_cols] = X[num_cols].apply(pd.to_numeric, errors="coerce")
    return X

def _apply_model_specific_inference_transforms(model_name, X_row):
    X = X_row.copy()
    if model_name == "Logistic (Fundamental)":
        for c in ["annual_inc", "revol_bal", "loan_amnt"]:
            if c in X.columns:
                vals = pd.to_numeric(X[c], errors="coerce")
                X[c] = np.log1p(vals.clip(lower=0))
    return X

def predict_from_user_input(model_name, form_data):
    spec = MODEL_SPECS[model_name]
    model = trained_bundle[spec["model_key"]]
    encoder = trained_bundle[spec["encoder_key"]]
    threshold = _get_human_loop_metrics(model_name)["threshold"]

    X_row = _build_input_row_for_encoder(encoder, form_data)
    X_row = _apply_model_specific_inference_transforms(model_name, X_row)
    X_enc = encoder.transform(X_row)

    p_default = float(model.predict_proba(X_enc)[:, 1][0])
    pred_class = int(p_default >= threshold)

    return {
        "prob_default": p_default,
        "threshold": threshold,
        "pred_class": pred_class,
        "risk_band": _risk_band(p_default),
    }

def _render_gauge(p, thr):
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=p,
        number={"valueformat": ".2%"},
        title={"text": "Predicted Default Probability"},
        gauge={
            "axis": {"range": [0, 1]},
            "bar": {"color": "crimson"},
            "steps": [
                {"range": [0.00, 0.05], "color": "#E8F5E9"},
                {"range": [0.05, 0.15], "color": "#FFF9C4"},
                {"range": [0.15, 0.30], "color": "#FFE0B2"},
                {"range": [0.30, 1.00], "color": "#FFCDD2"},
            ],
            "threshold": {"line": {"color": "black", "width": 3}, "thickness": 0.8, "value": thr},
        },
    ))
    fig.update_layout(template="plotly_white", height=260, margin=dict(l=10, r=10, t=70, b=10))
    return fig

In [4]:
# model summary
summary = trained_bundle.get("summary", None)
if summary is not None:
    display(summary.sort_values("val_pr_auc", ascending=False).reset_index(drop=True))
else:
    print("No summary table found in trained bundle.")

,model,val_roc_auc,val_pr_auc,val_brier,test_roc_auc,test_pr_auc,test_brier,val_best_thr_f1,val_best_f1,val_accuracy_at_best_f1_thr,val_precision_at_best_f1_thr,val_recall_at_best_f1_thr,test_accuracy_at_val_best_f1_thr,test_precision_at_val_best_f1_thr,test_recall_at_val_best_f1_thr,test_f1_at_val_best_f1_thr
0,XGB (Full),0.717182,0.155800,0.223519,0.712520,0.141501,0.220234,0.678941,0.230724,0.832342,0.167449,0.370865,0.842268,0.153577,0.346453,0.212816
1,XGB (Fundamental),0.672315,0.126593,0.219220,0.666881,0.110836,0.210933,0.598504,0.195307,0.790295,0.131990,0.375387,0.804704,0.118303,0.336806,0.175102
2,Logistic (Fundamental),0.625036,0.101383,0.081640,0.622717,0.090358,0.077727,0.202362,0.165183,0.656341,0.098875,0.501513,0.669054,0.089621,0.477991,0.150941


In [5]:
# Widgets + compact one-screen layout + callbacks
# UI Sizing 
def make_row(label_text, widget):
    widget.layout = widgets.Layout(width="150px", height="26px")
    label = widgets.HTML(
        f"<div style='width:220px; font-size:12px; line-height:1.2; font-weight:600;'>{label_text}</div>"
    )
    return widgets.HBox([label, widget], layout=widgets.Layout(width="100%", align_items="center"))

#  Widgets 
model_w = widgets.Dropdown(options=list(MODEL_SPECS.keys()), value="XGB (Fundamental)")
loan_amnt_w = widgets.FloatText(value=12000)
term_w = widgets.Dropdown(options=["36 months", "60 months"], value="36 months")
annual_inc_w = widgets.FloatText(value=75000)
emp_length_w = widgets.Dropdown(
    options=["< 1 year","1 year","2 years","3 years","4 years","5 years","6 years","7 years","8 years","9 years","10+ years"],
    value="5 years"
)
home_ownership_w = widgets.Dropdown(options=["RENT", "OWN", "MORTGAGE", "OTHER", "ANY", "NONE"], value="MORTGAGE")
verification_status_w = widgets.Dropdown(options=["Not Verified", "Source Verified", "Verified"], value="Verified")
purpose_w = widgets.Dropdown(
    options=[
        "debt_consolidation","credit_card","home_improvement","major_purchase","small_business",
        "car","medical","moving","vacation","house","wedding","renewable_energy","educational","other"
    ],
    value="debt_consolidation"
)
addr_state_w = widgets.Text(value="CA")

fico_low_w = widgets.IntText(value=680)
fico_high_w = widgets.IntText(value=684)
earliest_cr_line_w = widgets.Text(value="Jan-2010")

open_acc_w = widgets.IntText(value=10)
total_acc_w = widgets.IntText(value=24)
delinq_2yrs_w = widgets.IntText(value=0)
pub_rec_w = widgets.IntText(value=0)

revol_bal_w = widgets.FloatText(value=12000)
monthly_debt_w = widgets.FloatText(value=1500)
gross_monthly_income_w = widgets.FloatText(value=6250)
credit_limit_w = widgets.FloatText(value=25000)

dti_auto_w = widgets.FloatText(value=24.0, disabled=True)
revol_util_auto_w = widgets.FloatText(value=48.0, disabled=True)

int_rate_w = widgets.FloatText(value=5.5)
installment_w = widgets.FloatText(value=350.0)
grade_w = widgets.Dropdown(options=list("ABCDEFG"), value="A")
sub_grade_w = widgets.Dropdown(options=[f"{g}{i}" for g in "ABCDEFG" for i in range(1, 6)], value="A1")

predict_btn = widgets.Button(description="Predict", button_style="danger", icon="calculator", layout=widgets.Layout(width="110px"))
reset_btn = widgets.Button(description="Reset", icon="refresh", layout=widgets.Layout(width="110px"))

model_hint = widgets.HTML()
issue_date_hint = widgets.HTML(
    f"<span style='color:#555; font-size:12px;'><b>Issue date [issue_d]</b> auto-set to <b>{FIXED_ISSUE_DATE}</b>.</span>"
)
input_help = widgets.HTML("""
<div style='font-size:12px; color:#444; line-height:1.35;'>
<b>Quick definitions:</b><br>
- <b>Active accounts [open_acc]</b>: Number of active credit accounts (Too many = Risky).<br>
- <b>Total accounts [total_acc]</b>: Total number of credit history accounts (More history = Better).<br>
- <b>Delinquencies in 2 years [delinq_2yrs]</b>: Number of times you were late on credit payments in the last 2 years.<br>
- <b>Revolving balance ($) [revol_bal]</b>: Total amount you currently owe on revolving credit (like credit cards).<br>
- <b>Revolving credit limit ($)</b>: Total available revolving credit (your credit card limits combined).<br>
- <b>Debt-to-income (%) [dti]</b>: Percent of your monthly income used for debt payments.<br>
- <b>Revolving utilization (%) [revol_util]</b>: Percent of your revolving credit currently used.<br>
</div>
""")
out = widgets.Output()

full_only_widgets = [int_rate_w, installment_w, grade_w, sub_grade_w]

def _update_full_fields_enabled(*args):
    use_full = MODEL_SPECS[model_w.value]["uses_full_fields"]
    for w in full_only_widgets:
        w.disabled = not use_full
    model_hint.value = (
        "<span style='color:#0b5394; font-size:12px;'>Full-model fields are enabled.</span>"
        if use_full else
        "<span style='color:#666; font-size:12px;'>Full-model fields are disabled for this model.</span>"
    )

def _refresh_derived_preview(*args):
    try:
        dti, util = _compute_derived_financials(
            monthly_debt_w.value, gross_monthly_income_w.value, revol_bal_w.value, credit_limit_w.value
        )
        dti_auto_w.value = round(dti, 4)
        revol_util_auto_w.value = round(util, 4)
    except Exception:
        dti_auto_w.value = np.nan
        revol_util_auto_w.value = np.nan

model_w.observe(_update_full_fields_enabled, names="value")
_update_full_fields_enabled()

for w in [monthly_debt_w, gross_monthly_income_w, revol_bal_w, credit_limit_w]:
    w.observe(_refresh_derived_preview, names="value")
_refresh_derived_preview()

def _collect_form_data():
    dti_calc, util_calc = _compute_derived_financials(
        monthly_debt_w.value, gross_monthly_income_w.value, revol_bal_w.value, credit_limit_w.value
    )

    return {
        "loan_amnt": loan_amnt_w.value,
        "term": term_w.value,
        "annual_inc": annual_inc_w.value,
        "emp_length": emp_length_w.value,
        "home_ownership": home_ownership_w.value,
        "verification_status": verification_status_w.value,
        "purpose": purpose_w.value,
        "addr_state": addr_state_w.value.strip().upper() if isinstance(addr_state_w.value, str) else addr_state_w.value,

        "fico_range_low": fico_low_w.value,
        "fico_range_high": fico_high_w.value,
        "earliest_cr_line": earliest_cr_line_w.value.strip() if isinstance(earliest_cr_line_w.value, str) else earliest_cr_line_w.value,
        "issue_d": FIXED_ISSUE_DATE,

        "open_acc": open_acc_w.value,
        "total_acc": total_acc_w.value,
        "delinq_2yrs": delinq_2yrs_w.value,
        "pub_rec": pub_rec_w.value,

        "dti": dti_calc,
        "revol_bal": revol_bal_w.value,
        "revol_util": util_calc,

        "int_rate": int_rate_w.value,
        "installment": installment_w.value,
        "grade": grade_w.value,
        "sub_grade": sub_grade_w.value,
    }

def on_predict(_):
    with out:
        clear_output(wait=True)
        try:
            form = _collect_form_data()
            form_valid, warns = validate_form_by_model(model_w.value, form, clamp=CLAMP_OUT_OF_RANGE)

            print("Warning: Inputs outside realistic range may affect prediction.")
            if warns:
                print("Auto-adjustments:")
                for w in warns:
                    print(f"- {w}")

            result = predict_from_user_input(model_w.value, form_valid)
            p, thr, pred, band = result["prob_default"], result["threshold"], result["pred_class"], result["risk_band"]
            verdict = "LIKELY DEFAULT" if pred == 1 else "LIKELY NON-DEFAULT"

            print(f"Model: {model_w.value}")
            print(f"Issue date [issue_d] used internally: {FIXED_ISSUE_DATE}")
            print(f"Auto-calculated DTI [dti]: {form_valid['dti']:.4f}%")
            print(f"Auto-calculated Revolving utilization [revol_util]: {form_valid['revol_util']:.4f}%")
            print(f"Predicted default probability: {p:.4f} ({p:.2%})")
            print(f"Decision threshold: {thr:.4f}")
            print(f"Prediction: {verdict}")
            print(f"Risk band: {band}")

            m = _get_human_loop_metrics(model_w.value)
            print("\nHuman-in-the-loop disclaimer:")
            print(
                f"- Validation metrics at operating threshold: "
                f"F1={_fmt_metric(m['f1'])}, Accuracy={_fmt_metric(m['accuracy'])}, "
                f"Recall={_fmt_metric(m['recall'])}, Precision={_fmt_metric(m['precision'])}"
            )
            print("- This prediction is advisory only and must not be used as the sole basis for approval/denial.")

            display(_render_gauge(p, thr))

        except Exception as e:
            print("Prediction error:", str(e))
            print("Tips:")
            print("- Earliest credit line must be Mon-YYYY (e.g., Jan-2010)")
            print("- Gross monthly income and revolving credit limit must be > 0")

def on_reset(_):
    model_w.value = "XGB (Fundamental)"
    loan_amnt_w.value = 12000
    term_w.value = "36 months"
    annual_inc_w.value = 75000
    emp_length_w.value = "5 years"
    home_ownership_w.value = "MORTGAGE"
    verification_status_w.value = "Verified"
    purpose_w.value = "debt_consolidation"
    addr_state_w.value = "CA"
    fico_low_w.value = 680
    fico_high_w.value = 684
    earliest_cr_line_w.value = "Jan-2010"
    open_acc_w.value = 10
    total_acc_w.value = 24
    delinq_2yrs_w.value = 0
    pub_rec_w.value = 0
    revol_bal_w.value = 12000
    monthly_debt_w.value = 1500
    gross_monthly_income_w.value = 6250
    credit_limit_w.value = 25000
    int_rate_w.value = 5.5
    installment_w.value = 350.0
    grade_w.value = "A"
    sub_grade_w.value = "A1"
    _refresh_derived_preview()
    with out:
        clear_output(wait=True)
        print("Form reset complete.")

predict_btn.on_click(on_predict)
reset_btn.on_click(on_reset)

#  Pretty one-screen layout 
basic_section = widgets.VBox([
    issue_date_hint,
    make_row("Loan amount ($) [loan_amnt]", loan_amnt_w),
    make_row("Loan term [term]", term_w),
    make_row("Annual income ($) [annual_inc]", annual_inc_w),
    make_row("Employment length [emp_length]", emp_length_w),
    make_row("Home ownership [home_ownership]", home_ownership_w),
    make_row("Income verification [verification_status]", verification_status_w),
    make_row("Loan purpose [purpose]", purpose_w),
    make_row("Borrower state [addr_state]", addr_state_w),
], layout=widgets.Layout(padding="6px"))

credit_section = widgets.VBox([
    make_row("FICO range low [fico_range_low]", fico_low_w),
    make_row("FICO range high [fico_range_high]", fico_high_w),
    make_row("Earliest credit line [earliest_cr_line]", earliest_cr_line_w),
    make_row("Active accounts [open_acc]", open_acc_w),
    make_row("Total accounts [total_acc]", total_acc_w),
    make_row("Delinquencies in 2 years [delinq_2yrs]", delinq_2yrs_w),
    make_row("Public records [pub_rec]", pub_rec_w),
], layout=widgets.Layout(padding="6px"))

cashflow_section = widgets.VBox([
    make_row("Revolving balance ($) [revol_bal]", revol_bal_w),
    make_row("Monthly debt payments ($)", monthly_debt_w),
    make_row("Gross monthly income ($)", gross_monthly_income_w),
    make_row("Revolving credit limit ($)", credit_limit_w),
    make_row("Debt-to-income (%) [dti] (auto)", dti_auto_w),
    make_row("Revolving utilization (%) [revol_util] (auto)", revol_util_auto_w),
], layout=widgets.Layout(padding="6px"))

advanced_section = widgets.VBox([
    widgets.HTML("<b>Used only by XGB (Full)</b>"),
    model_hint,
    make_row("Interest rate (%) [int_rate]", int_rate_w),
    make_row("Installment amount ($) [installment]", installment_w),
    make_row("Loan grade [grade]", grade_w),
    make_row("Loan sub-grade [sub_grade]", sub_grade_w),
], layout=widgets.Layout(padding="6px"))

help_panel = widgets.Accordion(
    children=[widgets.VBox([input_help], layout=widgets.Layout(padding="4px"))],
    selected_index=None
)
help_panel.set_title(0, "Quick Definitions")

tabs = widgets.Tab(children=[basic_section, credit_section, cashflow_section, advanced_section])
tabs.set_title(0, "Borrower/Loan")
tabs.set_title(1, "Credit")
tabs.set_title(2, "Debt & Auto Calc")
tabs.set_title(3, "Full Model")
tabs.layout = widgets.Layout(width="100%", height="430px")

actions = widgets.HBox([predict_btn, reset_btn], layout=widgets.Layout(width="270px", justify_content="space-between", margin="6px 0 0 0"))

left_panel = widgets.VBox([make_row("Model", model_w), tabs, help_panel, actions], layout=widgets.Layout(width="50%", min_width="560px"))

out.layout = widgets.Layout(
    width="50%",
    min_width="520px",
    max_height="92vh",
    overflow_y="auto",
    border="1px solid #e2e2e2",
    padding="8px",
)

app = widgets.HBox([left_panel, out], layout=widgets.Layout(width="100%", align_items="flex-start", gap="12px"))
display(app)

### Notes
- Date fields must be `Mon-YYYY` (e.g., `Jan-2016`).
- Fundamental models ignore `interest rate`, `installment`, `grade`, `sub-grade`.
- XGB (Full) uses those extra fields.
- Model output is a risk indicator and should support (not replace) lending policy decisions.